# HX-Cascade v3 — step 1: dataset (CPU notebook)

Settings: **Accelerator: None** (the simulator is CPU-only; this keeps your GPU quota free), **Internet: On**.
Run with **Save Version > Save & Run All (Commit)** so it runs in the background for up to 12 hours.

What it does: random maps (split by map into train / val / test) plus the unmodified live map; every map is
simulated by the real engine headless, so the features are the published state (sensor noise, EnKF twin,
twin-model forecast) and the labels come from ground truth. The held-out splits are simulated first, then the
training maps one scenario at a time, and the run stops cleanly at `TIME_BUDGET_MIN` — so a time-out only
costs training breadth. `dataset.json` records what was actually produced.

Output: `/kaggle/working/cascade_v3/`. Use it in the training notebook with **Add Input > Your Work > this
notebook** (each committed version is kept), or turn it into a Kaggle Dataset from the Output tab.

**Sizing.** Local smoke run: one 720-cycle run ≈ 50 s of one core. On Kaggle's 4 cores expect roughly
150–250 runs per hour. The defaults plan 30 + 30 held-out maps × 8 + 8 live = 488 held-out runs, then up to
150 × 8 = 1200 training runs; whatever fits in the budget is kept. To add more, run this notebook again with
`DATASET_INPUT` pointing at the previous version's output: it resumes.

In [ ]:
# ============================ PARAMETERS ============================
# --- where the code comes from ----------------------------------------------
# "git": clone REPO_URL at REPO_REF (Settings > Internet must be ON)
# "dataset": the repo uploaded as a zip Kaggle Dataset, mounted at REPO_DATASET
CODE_SOURCE  = "git"
REPO_URL     = "https://github.com/GAURAV-MORE28/EventFlow-AI.git"
REPO_REF     = "gaurav"
REPO_COMMIT  = None        # pin an exact commit SHA here for a reproducible rerun
REPO_DATASET = "/kaggle/input/eventflow-ai/EventFlow-AI"

# --- dataset ------------------------------------------------------------------
MAPS_TRAIN, MAPS_VAL, MAPS_TEST = 150, 30, 30
SCENARIOS       = 8        # simulated runs per random map
LIVE_SCENARIOS  = 8        # runs on the unmodified live map (held-out test only)
STEPS           = 720      # cycles per run (720 x 30 s = 14:00-20:00 sim time)
SNAPSHOT_EVERY  = 10
DATA_SEED       = 7
WORKERS         = 4        # Kaggle CPU sessions have 4 cores
TIME_BUDGET_MIN = 630      # stop starting new runs after 10.5 h (session limit is 12 h)
DATASET_INPUT   = None     # resume: e.g. "/kaggle/input/<previous-version>/cascade_v3"
# =====================================================================

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/EventFlow-AI")          # outside /kaggle/working, so the code is not saved as output

# 1. The code, at a recorded commit.
if not REPO.exists():
    if CODE_SOURCE == "dataset":
        shutil.copytree(REPO_DATASET, REPO)
    else:
        url = REPO_URL
        try:   # private repo only: Add-ons > Secrets > GITHUB_TOKEN (never paste a token into a cell)
            from kaggle_secrets import UserSecretsClient
            url = url.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--quiet", "--branch", REPO_REF, url, str(REPO)], check=True)
        if REPO_COMMIT:
            subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True,
                            check=True).stdout.strip()
except Exception:
    COMMIT = REPO_COMMIT or None     # zip upload: no .git, so set REPO_COMMIT by hand
print("code commit:", COMMIT)

# 2. Dependencies. Kaggle ships torch and numpy: keep them, add the backend's
#    pinned packages (including torch_geometric) on top.
reqs = [l.split("#")[0].strip() for l in (REPO / "Backend" / "requirements.txt").read_text().splitlines()]
reqs = [r for r in reqs if r and r.split("==")[0].split("[")[0].lower() not in {"torch", "numpy", "pytest"}]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)

# 3. Environment check: PyG must work with THIS torch, and the forecaster must
#    run in the same mode as the live backend (no Chronos TSFM installed).
import torch, torch_geometric, numpy as np
from torch_geometric.nn import RGCNConv, TransformerConv
x = torch.randn(5, 8); ei = torch.tensor([[0, 1, 2], [1, 2, 3]])
TransformerConv(8, 4, heads=2, edge_dim=3)(x, ei, torch.randn(3, 3))
RGCNConv(8, 8, num_relations=2)(x, ei, torch.tensor([0, 1, 0]))
if importlib.util.find_spec("chronos") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "chronos-forecasting"], check=True)
    raise SystemExit("chronos was installed and has been removed: restart the kernel and run all again")
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | torch_geometric {torch_geometric.__version__} "
      f"| numpy {np.__version__} | cuda {torch.cuda.is_available()} | cores {os.cpu_count()}")
print("environment OK")

In [ ]:
DATA = WORK / "cascade_v3"
if DATASET_INPUT and not DATA.exists():
    shutil.copytree(DATASET_INPUT, DATA)      # inputs are read-only; resume in a copy
t0 = time.time()
subprocess.run([sys.executable, "-m", "scripts.cascade_dataset", "--out", str(DATA),
                "--maps-train", str(MAPS_TRAIN), "--maps-val", str(MAPS_VAL), "--maps-test", str(MAPS_TEST),
                "--scenarios", str(SCENARIOS), "--live-scenarios", str(LIVE_SCENARIOS), "--steps", str(STEPS),
                "--snapshot-every", str(SNAPSHOT_EVERY), "--seed", str(DATA_SEED), "--workers", str(WORKERS),
                "--time-budget-min", str(TIME_BUDGET_MIN)],
               cwd=REPO / "Backend", check=True)
print(f"dataset step {(time.time() - t0) / 3600:.1f} h")

In [ ]:
summary = json.loads((DATA / "dataset.json").read_text())
print(json.dumps({k: summary.get(k) for k in ("source_commit", "complete", "maps", "runs",
                                               "maps_rejected_by_simulator", "timing", "library_versions")}, indent=1))
size_mb = sum(f.stat().st_size for f in DATA.rglob("*") if f.is_file()) / 1e6
print(f"{size_mb:.0f} MB in {DATA}")